# Getting started with MIPY
A simulated longitudinal cohort with known true values: mask, impute, score.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import mipy
from mipy.datasets import simulate_longitudinal
from mipy.masking import Amputer
print('mipy', mipy.__version__)
mipy.list_methods()

mipy 2.0.0


,method,family,package,function,peer-reviewed,multiple imputation,uses patients,new in MIPY,available
0,Mean,Naive,scikit-learn,SimpleImputer(strategy='mean'),no,no,no,no,yes
1,Median,Naive,scikit-learn,SimpleImputer(strategy='median'),no,no,no,no,yes
2,LOCF,Longitudinal,pandas,groupby().ffill().bfill(),no,no,yes,no,yes
3,KNN,Neighbour,scikit-learn,KNNImputer,yes,no,no,no,yes
4,softImpute,Low-rank,mipy (numpy),"soft-thresholded SVD, CV lambda",yes,no,no,no,yes
5,MICE-Bayes,Chained,mipy (scikit-learn),BayesianRidge in FCS loop,yes,yes,no,no,yes
6,missForest,Chained,mipy (scikit-learn),ExtraTreesRegressor in FCS loop,yes,no,no,no,yes
7,sklearn-II,Chained,scikit-learn,IterativeImputer,no,no,no,no,yes
8,miceforest,Chained,miceforest,ImputationKernel().mice(),no,no,no,no,yes
9,HyperImpute,AutoML,hyperimpute,Imputers().get('hyperimpute'),yes,no,no,no,needs hyperimpute


In [2]:
sim = simulate_longitudinal(n_patients=80, n_visits=4, n_proteins=60, seed=1)
X, patient, time = sim['X'], sim['patient'], sim['time']
print(X.shape, 'complete truth')

(320, 60) complete truth


## Mask 20% of the values under MNAR (low values lost), impute, score

In [3]:
amp = Amputer(X, time=time, rate=0.2).ampute('MNAR', seed=7)
ctx = mipy.ImputeContext(patient=patient, time=time, mice_iter=3,
                         schema=mipy.build_variable_schema(amp['X']))
rows = []
for name in ['Mean', 'LOCF', 'softImpute', 'MICE-Bayes', 'MICE-Long', 'MICE-LongMNAR']:
    out, draws = mipy.run(name, amp['X'], ctx)
    ev = mipy.evaluate(X, out, amp['mask'], patient=patient)
    cov = mipy.predictive_coverage(np.where(amp['mask'], X, np.nan), draws, amp['mask'])['coverage'] if draws else np.nan
    rows.append({'method': name, 'nRMSE': ev['NRMSE_median'], 'bias': ev['bias_median'],
                 'SD ratio': ev['structure_ratio'], 'MI coverage': cov})
pd.DataFrame(rows).round(3)

,method,nRMSE,bias,SD ratio,MI coverage
0,Mean,2.034,1.173,0.840,NaN
1,LOCF,1.541,0.646,0.700,NaN
2,softImpute,1.562,0.901,0.788,NaN
3,MICE-Bayes,1.345,0.668,0.853,0.865
4,MICE-Long,1.159,0.518,0.786,0.883
5,MICE-LongMNAR,0.992,0.334,0.813,0.908


## The same through the data-frame interface

In [4]:
df = mipy.io.from_matrix(amp['X'], patient, time, sim['proteins'])
done = mipy.impute(df, method='MICE-Long', id_col='patient_id', time_col='visit', mice_iter=3)
five = mipy.impute(df, method='MICE-Long', id_col='patient_id', time_col='visit', m=5, mice_iter=3)
print(done.shape, len(five), int(done[sim['proteins']].isna().sum().sum()), 'missing left')

(320, 62) 5 0 missing left


## Benchmark on your own data

In [5]:
full = mipy.io.from_matrix(X, patient, time, sim['proteins'])
res = mipy.benchmark(full, methods=['Mean', 'softImpute', 'MICE-Long'], id_col='patient_id',
                     time_col='visit', mechanisms=('MCAR', 'MNAR'), repeats=2, m=3)
res.pivot_table(index='method', columns='mechanism', values='NRMSE').round(3)

mechanism,MCAR,MNAR
method,,
MICE-Long,0.717,1.246
Mean,1.004,2.035
softImpute,0.713,1.614


## The evidence behind Python imputation packages

In [6]:
mipy.literature()[['package', 'package_paper', 'peer_reviewed']]

,package,package_paper,peer_reviewed
0,scikit-learn,"Pedregosa et al. 2011, JMLR 12:2825-2830 (gene...",library yes; IterativeImputer no
1,miceforest,none found,no
2,hyperimpute,"Jarrett et al. 2022, ICML, PMLR 162:9916-9937",yes
3,fancyimpute,none (software DOI 10.5281/zenodo.51773),methods only
4,missingpy,none,methods only
5,missforest,none (software DOI 10.5281/zenodo.13368883),methods only
6,statsmodels,"Seabold & Perktold 2010, Proc. SciPy, 92-96 (g...",library only
7,autoimpute,none found,no
8,impyute,none found,no
9,datawig,"Biessmann et al. 2019, JMLR 20(175):1-6; Biess...",yes
